# 跨池路徑規劃模擬教學

本 notebook 展示如何使用最新的跨池模擬功能，包含：
1. 單池模擬（向後相容）
2. 多池路徑規劃
3. 重複稀有追蹤

In [1]:
import sys
import os
sys.path.append(os.path.abspath("../src"))

from battle_cat_calc.models import GachaSlot, Position, Track, GachaPool, GachaStep
from battle_cat_calc.simulator import GachaSimulator

## 1. 單池模擬（向後相容）

舊式 API 仍然完全支援：

In [2]:
# 建立 Track A 和 Track B
track_a = Track("A", [
    GachaSlot(id=Position(1, 'A'), result="貓劍客", guaranteed="地藏貓", next_pos=Position(11, 'B')),
    GachaSlot(id=Position(2, 'A'), result="鬥士貓", guaranteed="地藏貓", next_pos=Position(12, 'B')),
    GachaSlot(id=Position(3, 'A'), result="海盜貓", guaranteed="地藏貓", next_pos=Position(13, 'B')),
    GachaSlot(id=Position(4, 'A'), result="弓箭貓", guaranteed="地藏貓", next_pos=Position(14, 'B')),
    GachaSlot(id=Position(5, 'A'), result="狙擊貓", guaranteed="地藏貓", next_pos=Position(15, 'B')),
])

track_b = Track("B", [
    GachaSlot(id=Position(11, 'B'), result="影傑貓", guaranteed="御鯨貓", next_pos=None),
    GachaSlot(id=Position(12, 'B'), result="忍者貓", guaranteed="御鯨貓", next_pos=None),
])

# 舊式 API：直接傳入 track_a, track_b
sim = GachaSimulator(track_a, track_b)
print(f"單池模式: {not sim.is_multi_pool}")
print(f"池子列表: {list(sim.pools.keys())}")

單池模式: True
池子列表: ['default']


In [3]:
# 尋找最短路徑到達「海盜貓」
result = sim.find_best_path(
    start_pos=Position(1, 'A'),
    target_cats={"海盜貓"},
    max_rolls=10
)

print(f"總花費: {result['total_cost']} 抽")
print(f"\n路徑:")
for step in result['path']:
    print(f"  {step.source.id}{step.source.track} → {step.target.id}{step.target.track}: {step.unit}")

總花費: 3 抽

路徑:
  1A → 2A: 貓劍客
  2A → 3A: 鬥士貓
  3A → 4A: 海盜貓


## 2. 多池路徑規劃

新功能：同時考慮多個轉蛋池，找出最佳路徑：

In [4]:
# 池 A：超靈魂祭（有地藏、佛陀、御鯨等）
ultra_souls = GachaPool(
    pool_id="ultra_souls",
    name="超靈魂祭",
    track_a=Track("A", [
        GachaSlot(id=Position(1, 'A'), result="貓劍客", guaranteed="地藏貓", next_pos=Position(11, 'B')),
        GachaSlot(id=Position(2, 'A'), result="鬥士貓", guaranteed="地藏貓", next_pos=Position(12, 'B')),
        GachaSlot(id=Position(3, 'A'), result="海盜貓", guaranteed="佛陀貓", next_pos=Position(13, 'B')),
        GachaSlot(id=Position(4, 'A'), result="弓箭貓", guaranteed="御鯨貓", next_pos=Position(14, 'B')),
        GachaSlot(id=Position(5, 'A'), result="狙擊貓", guaranteed="地藏貓", next_pos=Position(15, 'B')),
    ]),
    track_b=Track("B", [
        GachaSlot(id=Position(11, 'B'), result="B11", guaranteed="G", next_pos=None),
    ]),
    has_guaranteed=True,
    guaranteed_cost=10
)

# 池 B：極祭典（有 Mitama、D'arktanyan 等，但沒有必中）
epicfest = GachaPool(
    pool_id="epicfest",
    name="極祭典",
    track_a=Track("A", [
        GachaSlot(id=Position(1, 'A'), result="貓劍客", guaranteed="Mitama", next_pos=Position(11, 'B')),
        GachaSlot(id=Position(2, 'A'), result="Mitama", guaranteed="Mitama", next_pos=Position(12, 'B')),
        GachaSlot(id=Position(3, 'A'), result="海盜貓", guaranteed="D'arkt", next_pos=Position(13, 'B')),
        GachaSlot(id=Position(4, 'A'), result="弓箭貓", guaranteed="D'arkt", next_pos=Position(14, 'B')),
        GachaSlot(id=Position(5, 'A'), result="狙擊貓", guaranteed="Mitama", next_pos=Position(15, 'B')),
    ]),
    track_b=Track("B", []),
    has_guaranteed=False  # 極祭典沒有必中
)

print("已建立兩個池：")
print(f"  - {ultra_souls.name} (has_guaranteed={ultra_souls.has_guaranteed})")
print(f"  - {epicfest.name} (has_guaranteed={epicfest.has_guaranteed})")

已建立兩個池：
  - 超靈魂祭 (has_guaranteed=True)
  - 極祭典 (has_guaranteed=False)


In [5]:
# 多池模擬器
multi_sim = GachaSimulator([ultra_souls, epicfest])

print(f"多池模式: {multi_sim.is_multi_pool}")
print(f"池子列表: {list(multi_sim.pools.keys())}")

多池模式: True
池子列表: ['ultra_souls', 'epicfest']


In [6]:
# 尋找 Mitama（只在 epicfest 的 2A 有）
result = multi_sim.find_best_path(
    start_pos=Position(1, 'A'),
    target_cats={"Mitama"},
    max_rolls=10
)

print(f"目標: Mitama")
print(f"總花費: {result['total_cost']} 抽")
print(f"\n路徑:")
for step in result['path']:
    pool_name = multi_sim.pools[step.pool_id].name
    print(f"  [{pool_name}] {step.source.id}{step.source.track} → {step.target.id}{step.target.track}: {step.unit}")

目標: Mitama
總花費: 2 抽

路徑:
  [超靈魂祭] 1A → 2A: 貓劍客
  [極祭典] 2A → 3A: Mitama


## 3. GachaStep 詳細資訊

每個步驟都包含詳細的上下文資訊：

In [7]:
if result:
    for i, step in enumerate(result['path']):
        print(f"Step {i+1}:")
        print(f"  - 來源: {step.source}")
        print(f"  - 目標: {step.target}")
        print(f"  - 獲得單位: {step.unit}")
        print(f"  - 池子 ID: {step.pool_id}")
        print(f"  - 是否保底: {step.is_guaranteed}")
        print(f"  - 是否跳軌: {step.is_track_switch}")
        print(f"  - 重複稀有: {step.is_duplicate_rare}")
        print(f"  - 上下文: {step.context_data}")
        print()

Step 1:
  - 來源: Position(id=1, track='A')
  - 目標: Position(id=2, track='A')
  - 獲得單位: 貓劍客
  - 池子 ID: ultra_souls
  - 是否保底: False
  - 是否跳軌: False
  - 重複稀有: False
  - 上下文: {'last_rolled_rare': None}

Step 2:
  - 來源: Position(id=2, track='A')
  - 目標: Position(id=3, track='A')
  - 獲得單位: Mitama
  - 池子 ID: epicfest
  - 是否保底: False
  - 是否跳軌: False
  - 重複稀有: False
  - 上下文: {'last_rolled_rare': None}



## 4. 使用保底跳軌

透過保底前往 Track B 的目標：

In [8]:
# 建立一個 Track B 有特殊目標的場景
pool_with_b = GachaPool(
    pool_id="test_pool",
    name="測試池",
    track_a=Track("A", [
        GachaSlot(id=Position(i, 'A'), result=f"A{i}", guaranteed="保底貓", next_pos=Position(i+10, 'B'))
        for i in range(1, 15)
    ]),
    track_b=Track("B", [
        GachaSlot(id=Position(11, 'B'), result="B11", guaranteed="G", next_pos=None),
        GachaSlot(id=Position(12, 'B'), result="目標貓", guaranteed="G", next_pos=None),
        GachaSlot(id=Position(13, 'B'), result="B13", guaranteed="G", next_pos=None),
    ]),
    has_guaranteed=True
)

sim_b = GachaSimulator([pool_with_b])

# 尋找「目標貓」（在 12B）
result_b = sim_b.find_best_path(
    start_pos=Position(1, 'A'),
    target_cats={"目標貓"},
    max_rolls=20
)

print(f"目標: 目標貓 (位於 12B)")
print(f"總花費: {result_b['total_cost']} 抽")
print(f"\n路徑:")
for step in result_b['path']:
    gua_text = "(保底)" if step.is_guaranteed else ""
    switch_text = "→跳軌" if step.is_track_switch else ""
    print(f"  {step.source.id}{step.source.track} → {step.target.id}{step.target.track}: {step.unit} {gua_text}{switch_text}")

目標: 目標貓 (位於 12B)
總花費: 12 抽

路徑:
  1A → 11B: 保底貓 (保底)→跳軌
  11B → 12B: B11 
  12B → 13B: 目標貓 


## 5. 搭配實際資料使用

從 Godfat 抓取資料後進行模擬：

```python
from battle_cat_calc.fetcher import BCDataFetcher
from battle_cat_calc.parser import BCParser

# 抓取資料
fetcher = BCDataFetcher()
html = await fetcher.fetch_html(seed="1234567890", event="ultra_souls", lang="tw", count=100)

# 解析
parser = BCParser(html)
track_a = parser.track_a
track_b = parser.track_b

# 建立池
pool = GachaPool(
    pool_id="ultra_souls",
    name="超靈魂祭",
    track_a=track_a,
    track_b=track_b,
    has_guaranteed=True
)

# 模擬
sim = GachaSimulator([pool])
result = sim.find_best_path(Position(1, 'A'), {"地藏貓", "御鯨貓"}, 100)
```